In [ ]:
# Cell 1: setup and data

# v3 step 4a: load the v3 data onto the GPU.
import json, time
from pathlib import Path
import numpy as np, pandas as pd, torch
from torch import nn
from google.colab import drive

drive.mount("/content/drive")
assert torch.cuda.is_available(), "Runtime > Change runtime type > T4 GPU"
DEVICE = "cuda"
DATA_DIR = Path("/content/drive/MyDrive/tastelet/data/processed_v3")
CKPT_PATH = DATA_DIR / "training_checkpoint.pt"  # resume point if Colab disconnects
MODEL_PATH = DATA_DIR / "ncf_model_v3.pt"        # best model, in the format the backend loads

SEED = 42
EMBEDDING_DIM, DROPOUT, LR = 32, 0.2, 1e-2  # same as v1 (CLAUDE.md fixed decisions)
N_NEGATIVES, NEGATIVE_RATING, POPULARITY_EXPONENT = 4, 0.5, 0.75  # ADR 0001 + 0002
BATCH_SIZE = 16_384  # big batches keep the GPU busy; v1 data was 300x smaller
EPOCHS = 20
TOP_K, LIKED_RATING, N_EVAL_USERS = 10, 4.0, 5_000

torch.manual_seed(SEED)
train = pd.read_parquet(DATA_DIR / "ratings_train.parquet", columns=["user_idx", "movie_idx", "rating"])
val = pd.read_parquet(DATA_DIR / "ratings_val.parquet", columns=["user_idx", "movie_idx", "rating"])
test = pd.read_parquet(MODEL_PATH.parent / "ratings_test.parquet")
n_users = len(json.loads((DATA_DIR / "user2idx.json").read_text()))
n_movies = len(json.loads((DATA_DIR / "movie2idx.json").read_text()))

pos_users = torch.tensor(train["user_idx"].values, device=DEVICE)
pos_movies = torch.tensor(train["movie_idx"].values, device=DEVICE)
pos_labels = torch.tensor(train["rating"].values, device=DEVICE)
rating_counts = np.bincount(train["movie_idx"], minlength=n_movies)
# Negatives drawn proportional to count^0.75 (ADR 0002), via a cumulative distribution (fast on GPU).
negative_cdf = torch.tensor(np.cumsum(rating_counts**POPULARITY_EXPONENT), device=DEVICE, dtype=torch.float64)
negative_cdf /= negative_cdf[-1].clone()
print(f"{len(train):,} train ratings · {n_users:,} users · {n_movies:,} movies on {torch.cuda.get_device_name()}")


class NCF(nn.Module):  # identical to backend/app/ml/model.py, so the backend can load the weights
    def __init__(self, n_users: int, n_movies: int, dim: int, dropout: float) -> None:
        super().__init__()
        self.user_embedding = nn.Embedding(n_users, dim)
        self.movie_embedding = nn.Embedding(n_movies, dim)
        self.mlp = nn.Sequential(
            nn.Linear(2 * dim, 64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 32), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(32, 1),
        )

    def score_vec(self, user_vecs, movie_ids):
        return self.mlp(torch.cat([user_vecs, self.movie_embedding(movie_ids)], dim=1)).squeeze(-1)

    def forward(self, user_ids, movie_ids):
        return self.score_vec(self.user_embedding(user_ids), movie_ids)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


AssertionError: Runtime > Change runtime type > T4 GPU

In [ ]:
# Cell 2: train, with Hit@10 and a checkpoint after every epoch

# v3 step 4b: train the ranking model; resume from the checkpoint if one exists.
rng = np.random.default_rng(SEED)
liked_val = val[val["rating"] >= LIKED_RATING]
eval_users = rng.choice(np.sort(liked_val["user_idx"].unique()), N_EVAL_USERS, replace=False)
liked_by_user = liked_val[liked_val["user_idx"].isin(eval_users)].groupby("user_idx")["movie_idx"].apply(set)
seen_by_user = train[train["user_idx"].isin(eval_users)].groupby("user_idx")["movie_idx"].apply(np.array)
all_movies = torch.arange(n_movies, device=DEVICE)


@torch.no_grad()
def hit_at_10(score_user, users=eval_users) -> float:

    """Share of 5,000 sampled val users with a liked val movie in their top 10 (train movies excluded)."""
    hits = 0
    for user in users:
        scores = score_user(user)
        scores[torch.as_tensor(seen_by_user.get(user, []), device=DEVICE, dtype=torch.long)] = -torch.inf
        hits += bool(liked_by_user[user] & set(torch.topk(scores, TOP_K).indices.tolist()))
    return hits / len(users)


popularity = torch.tensor(rating_counts, device=DEVICE, dtype=torch.float32)
print("popularity val Hit@10:", hit_at_10(lambda _: popularity.clone()))

model = NCF(n_users, n_movies, EMBEDDING_DIM, DROPOUT).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
start_epoch, best_hit = 0, -1.0
if CKPT_PATH.exists():  # Colab dropped the session: continue where we stopped
    ckpt = torch.load(CKPT_PATH)
    model.load_state_dict(ckpt["model"]); optimizer.load_state_dict(ckpt["optimizer"])
    start_epoch, best_hit = ckpt["epoch"] + 1, ckpt["best_hit"]
    print(f"resumed after epoch {ckpt['epoch']}, best val Hit@10 {best_hit:.4f}")

n_pos = len(pos_users)
for epoch in range(start_epoch, EPOCHS):
    started = time.time()
    gen = torch.Generator(device=DEVICE).manual_seed(SEED + epoch)  # new negatives each epoch, reproducible
    neg_movies = torch.searchsorted(negative_cdf, torch.rand(n_pos * N_NEGATIVES, device=DEVICE, generator=gen, dtype=torch.float64))
    users = torch.cat([pos_users, pos_users.repeat(N_NEGATIVES)])
    movies = torch.cat([pos_movies, neg_movies.int()])
    labels = torch.cat([pos_labels, torch.full((n_pos * N_NEGATIVES,), NEGATIVE_RATING, device=DEVICE)])
    order = torch.randperm(len(users), device=DEVICE, generator=gen)

    model.train()
    total_loss = 0.0
    for start in range(0, len(order), BATCH_SIZE):
        batch = order[start : start + BATCH_SIZE]
        optimizer.zero_grad()
        loss = ((model(users[batch].long(), movies[batch].long()) - labels[batch]) ** 2).mean()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(batch)
    del users, movies, labels, order, neg_movies  # free GPU memory before evaluating

    model.eval()
    val_hit = hit_at_10(lambda u: model(torch.full((n_movies,), int(u), device=DEVICE), all_movies))
    print(f"epoch {epoch}: loss {total_loss / (n_pos * (1 + N_NEGATIVES)):.4f} · val Hit@10 {val_hit:.4f} · {time.time() - started:.0f}s")
    if val_hit > best_hit:  # keep the best epoch, in the exact format backend/app/ml/model.py loads
        best_hit = val_hit
        torch.save({"n_users": n_users, "n_movies": n_movies, "embedding_dim": EMBEDDING_DIM,
                    "dropout": DROPOUT, "state_dict": {k: v.cpu() for k, v in model.state_dict().items()}}, MODEL_PATH)
    torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                "epoch": epoch, "best_hit": best_hit}, CKPT_PATH)
print("best val Hit@10:", best_hit)

In [ ]:
# Cell 3: test Hit@10 of the best epoch vs popularity, on the same 5,000 test users
rng = np.random.default_rng(SEED)
liked_test = test[test["rating"] >= LIKED_RATING]
test_users = rng.choice(np.sort(liked_test["user_idx"].unique()), N_EVAL_USERS, replace=False)
liked_test_by_user = liked_test[liked_test["user_idx"].isin(test_users)].groupby("user_idx")["movie_idx"].apply(set)
seen_before_test = pd.concat([train, val])  # by test time the user has watched train AND val movies
seen_test_by_user = (seen_before_test[seen_before_test["user_idx"].isin(test_users)]
                     .groupby("user_idx")["movie_idx"].apply(np.array))


@torch.no_grad()
def test_hit_at_10(score_user, users=test_users) -> float:
    """Same rule as hit_at_10, on the test split."""
    hits = 0
    for user in users:
        scores = score_user(user)
        scores[torch.as_tensor(seen_test_by_user.get(user, []), device=DEVICE, dtype=torch.long)] = -torch.inf
        hits += bool(liked_test_by_user[user] & set(torch.topk(scores, TOP_K).indices.tolist()))
    return hits / len(users)


saved = torch.load(MODEL_PATH)
best_model = NCF(n_users, n_movies, EMBEDDING_DIM, DROPOUT).to(DEVICE)
best_model.load_state_dict(saved["state_dict"])
best_model.eval()

print("popularity test Hit@10:", test_hit_at_10(lambda _: popularity.clone()))
print("NCF test Hit@10:", test_hit_at_10(lambda u: best_model(torch.full((n_movies,), int(u), device=DEVICE), all_movies)))

In [ ]:
# Cell 4: cold start — fold in a new user from 5 picks, the same way the backend does
FOLD_IN_STEPS, FOLD_IN_LR, FOLD_IN_NEGATIVES, FOLD_IN_L2 = 100, 0.05, 20, 1e-3  # backend/app/core/config.py
PICK_RATING, N_COLD_START_PICKS, N_COLD_START_USERS = 5.0, 5, 1_000

cold_users = test_users[:N_COLD_START_USERS]  # test_users is already a random sample
cold_train = train[train["user_idx"].isin(cold_users)]
picks_by_user = {u: g.nlargest(N_COLD_START_PICKS, "rating")["movie_idx"].tolist()
                 for u, g in cold_train.groupby("user_idx")}
negative_weights = rating_counts.astype(float) ** POPULARITY_EXPONENT  # ADR 0002
best_model.requires_grad_(False)  # freeze the model; only the user vector learns
mean_user_vec = best_model.user_embedding.weight.mean(dim=0, keepdim=True)


def fold_in_user_vector(picks: list[int], steps: int = FOLD_IN_STEPS, lr: float = FOLD_IN_LR,
                        l2: float = FOLD_IN_L2, start: torch.Tensor | None = None) -> torch.Tensor:
    """Same as the backend: picks -> 5.0, 20 popularity-sampled other movies -> 0.5, Adam steps from `start`."""
    rng = np.random.default_rng(SEED)  # same picks -> same vector, like the app
    user_vec = nn.Parameter((mean_user_vec if start is None else start).clone())
    optimizer = torch.optim.Adam([user_vec], lr=lr)
    unpicked = np.setdiff1d(np.arange(n_movies), picks)
    probs = negative_weights[unpicked] / negative_weights[unpicked].sum()
    negatives = rng.choice(unpicked, size=FOLD_IN_NEGATIVES, replace=False, p=probs)
    movie_ids = torch.tensor(np.concatenate([picks, negatives]), device=DEVICE, dtype=torch.long)
    targets = torch.tensor([PICK_RATING] * len(picks) + [NEGATIVE_RATING] * FOLD_IN_NEGATIVES, device=DEVICE)
    for _ in range(steps):
        optimizer.zero_grad()
        predicted = best_model.score_vec(user_vec.expand(len(movie_ids), -1), movie_ids)
        loss = ((predicted - targets) ** 2).mean() + l2 * (user_vec**2).sum()
        loss.backward()
        optimizer.step()
    return user_vec.detach()


started = time.time()
cold_vecs = {u: fold_in_user_vector(picks_by_user[u]) for u in cold_users}  # outside no_grad: fold-in needs gradients
print(f"folded in {len(cold_vecs):,} users in {time.time() - started:.0f}s")

print("popularity:", test_hit_at_10(lambda _: popularity.clone(), cold_users))
print("NCF, full history:", test_hit_at_10(lambda u: best_model(torch.full((n_movies,), int(u), device=DEVICE), all_movies), cold_users))
print("NCF, cold start (5 picks):", test_hit_at_10(lambda u: best_model.score_vec(cold_vecs[u].expand(n_movies, -1), all_movies), cold_users))

In [ ]:
# Cell 5: diagnose — how popular and how varied are the top-10 lists?
@torch.no_grad()
def top10_movies(score_user, users) -> np.ndarray:
    lists = []
    for user in users:
        scores = score_user(user)
        scores[torch.as_tensor(seen_test_by_user.get(user, []), device=DEVICE, dtype=torch.long)] = -torch.inf
        lists.append(torch.topk(scores, TOP_K).indices.cpu().numpy())
    return np.concatenate(lists)


def full_history_scores(u):
    return best_model(torch.full((n_movies,), int(u), device=DEVICE), all_movies)


def cold_start_scores(u):
    return best_model.score_vec(cold_vecs[u].expand(n_movies, -1), all_movies)


for name, score_user in [("full history", full_history_scores), ("cold start", cold_start_scores)]:
    movies = top10_movies(score_user, cold_users)
    print(f"{name}: median rating count {np.median(rating_counts[movies]):,.0f} · "
          f"{len(np.unique(movies)):,} different movies in {len(movies):,} slots")
all_picks = np.concatenate(list(picks_by_user.values()))
print(f"picks: median rating count {np.median(rating_counts[all_picks]):,.0f}")

In [ ]:
# Cell 6: did the fold-in finish learning? picks should score ~5.0
with torch.no_grad():
    pick_scores = [best_model.score_vec(cold_vecs[u].expand(N_COLD_START_PICKS, -1),
                                        torch.tensor(picks_by_user[u], device=DEVICE)).mean().item()
                   for u in cold_users]
    folded_norms = [cold_vecs[u].norm().item() for u in cold_users]
    trained_norms = best_model.user_embedding.weight[torch.as_tensor(cold_users, device=DEVICE)].norm(dim=1)
print(f"pick scores: median {np.median(pick_scores):.2f} (target {PICK_RATING})")
print(f"user vector length: folded-in median {np.median(folded_norms):.2f} · trained median {trained_norms.median().item():.2f}")

In [ ]:
# Cell 7: tune the fold-in on VAL users (never test); confirm only the winner on test afterwards
SWEEP_USERS = eval_users[:300]  # val sample from Cell 2
sweep_train = train[train["user_idx"].isin(SWEEP_USERS)]
sweep_picks = {u: g.nlargest(N_COLD_START_PICKS, "rating")["movie_idx"].tolist()
               for u, g in sweep_train.groupby("user_idx")}
FOLD_IN_SETUPS = {  # name: (steps, lr, l2)
    "now: 100 steps, lr 0.05, L2 1e-3": (100, 0.05, 1e-3),
    "300 steps": (300, 0.05, 1e-3),
    "300 steps, lr 0.1": (300, 0.1, 1e-3),
    "300 steps, L2 1e-4": (300, 0.05, 1e-4),
}

for name, (steps, lr, l2) in FOLD_IN_SETUPS.items():
    started = time.time()
    vecs = {u: fold_in_user_vector(sweep_picks[u], steps, lr, l2) for u in SWEEP_USERS}
    with torch.no_grad():
        pick_score = np.median([best_model.score_vec(vecs[u].expand(N_COLD_START_PICKS, -1),
                                                     torch.tensor(sweep_picks[u], device=DEVICE)).mean().item()
                                for u in SWEEP_USERS])
    hit = hit_at_10(lambda u: best_model.score_vec(vecs[u].expand(n_movies, -1), all_movies), SWEEP_USERS)
    print(f"{name}: val Hit@10 {hit:.3f} · pick score {pick_score:.2f} · {time.time() - started:.0f}s")
print("popularity:", hit_at_10(lambda _: popularity.clone(), SWEEP_USERS))

In [ ]:
import pyarrow.parquet as pq
print(pq.read_schema(DATA_DIR / "ratings_train.parquet").names)

In [ ]:
# Cell 8: does WHICH 5 picks matter? Same 300 val users and fold-in as Cell 7 (old rule: 0.170)
sweep_ratings = pd.read_parquet(DATA_DIR / "ratings_train.parquet",
                                columns=["user_idx", "movie_idx", "rating", "timestamp"],
                                filters=[("user_idx", "in", SWEEP_USERS.tolist())])
liked_sweep = sweep_ratings[sweep_ratings["rating"] >= LIKED_RATING]
PICK_RULES = {
    "most recent likes": lambda g: g.nlargest(N_COLD_START_PICKS, "timestamp"),
    "random likes": lambda g: g.sample(min(N_COLD_START_PICKS, len(g)), random_state=SEED),
}

for name, pick_rule in PICK_RULES.items():
    picks = {u: pick_rule(g)["movie_idx"].tolist() for u, g in liked_sweep.groupby("user_idx")}
    picks = {u: picks.get(u, sweep_picks[u]) for u in SWEEP_USERS}  # no liked train movie: keep the old picks
    vecs = {u: fold_in_user_vector(picks[u]) for u in SWEEP_USERS}
    hit = hit_at_10(lambda u: best_model.score_vec(vecs[u].expand(n_movies, -1), all_movies), SWEEP_USERS)
    pick_counts = rating_counts[np.concatenate(list(picks.values()))]
    print(f"{name}: val Hit@10 {hit:.3f} · picks median rating count {np.median(pick_counts):,.0f}")

In [ ]:
# Cell 9: the app as shipped = fold-in + "prefer newer" boost (ADR 0004), same 300 val users
movies = pd.read_csv("/content/drive/MyDrive/tastelet/data/raw/ml-32m/movies.csv")
movie2idx = json.loads((DATA_DIR / "movie2idx.json").read_text())
title_years = movies["title"].str.extract(r"\((\d{4})\)\s*$", expand=False).astype(float)
movie_years = np.full(n_movies, np.nan)
for movie_id, year in zip(movies["movieId"], title_years):
    idx = movie2idx.get(str(movie_id))
    if idx is not None:
        movie_years[idx] = year
print(f"{np.isnan(movie_years).sum():,} of {n_movies:,} movies without a year (they get no boost) · newest {np.nanmax(movie_years):.0f}")

FRESHNESS_FROM_YEAR, FRESHNESS_TO_YEAR = 1980, int(np.nanmax(movie_years))  # backend: 1980 to the catalog's last year
freshness = np.clip((movie_years - FRESHNESS_FROM_YEAR) / (FRESHNESS_TO_YEAR - FRESHNESS_FROM_YEAR), 0, 1)
freshness = torch.tensor(np.nan_to_num(freshness), device=DEVICE, dtype=torch.float32)

old_rule_vecs = {u: fold_in_user_vector(sweep_picks[u]) for u in SWEEP_USERS}  # Cell 7 picks, default fold-in
for boost in [0.0, 0.3, 0.6, 1.0]:  # 0.0 must repeat Cell 7's 0.170; 0.3 is the app's NEWER_BOOST
    hit = hit_at_10(lambda u: best_model.score_vec(old_rule_vecs[u].expand(n_movies, -1), all_movies)
                    + boost * freshness, SWEEP_USERS)
    print(f"boost {boost}: val Hit@10 {hit:.3f}")

In [ ]:
# Cell 10: popularity blend for cold start, tuned on the same 300 val users
popularity_scaled = torch.log1p(popularity) / torch.log1p(popularity).max()  # 0..1; log tames the skew
for w in [0.0, 0.5, 1.0, 2.0, 4.0, 8.0]:  # 0.0 repeats 0.170; very large w turns into plain popularity (0.207)
    hit = hit_at_10(lambda u: best_model.score_vec(old_rule_vecs[u].expand(n_movies, -1), all_movies)
                    + w * popularity_scaled, SWEEP_USERS)
    print(f"w {w}: val Hit@10 {hit:.3f}")

In [ ]:
# Cell 11: find where the blend peaks, on 1,000 val users (noise ±0.013) instead of 300
TUNE_USERS = eval_users[:1_000]
tune_train = train[train["user_idx"].isin(TUNE_USERS)]
tune_picks = {u: g.nlargest(N_COLD_START_PICKS, "rating")["movie_idx"].tolist()
              for u, g in tune_train.groupby("user_idx")}
tune_vecs = {u: fold_in_user_vector(tune_picks[u]) for u in TUNE_USERS}

print("popularity:", hit_at_10(lambda _: popularity.clone(), TUNE_USERS))
for w in [0.0, 4.0, 8.0, 16.0, 32.0, 100.0]:  # 100 should be almost plain popularity
    hit = hit_at_10(lambda u: best_model.score_vec(tune_vecs[u].expand(n_movies, -1), all_movies)
                    + w * popularity_scaled, TUNE_USERS)
    print(f"w {w}: val Hit@10 {hit:.3f}")

In [ ]:
# Cell 12: final check ON TEST (once) with w chosen on val, plus how personal the lists stay
BLEND_W = 16.0  # peak of the val sweep (Cell 11)


def blended_cold_scores(u):
    return cold_start_scores(u) + BLEND_W * popularity_scaled


print("popularity:", test_hit_at_10(lambda _: popularity.clone(), cold_users))
print("cold start, no blend:", test_hit_at_10(cold_start_scores, cold_users))  # control: must repeat 0.159
print("cold start, blend:", test_hit_at_10(blended_cold_scores, cold_users))

for name, score_user in [("popularity", lambda _: popularity.clone()), ("no blend", cold_start_scores),
                         ("blend", blended_cold_scores), ("full history", full_history_scores)]:
    movies = top10_movies(score_user, cold_users)
    print(f"{name}: {len(np.unique(movies)):,} different movies in {len(movies):,} slots · "
          f"median rating count {np.median(rating_counts[movies]):,.0f}")

In [ ]:
# Cell 13: how personal do the lists stay at each w? (no test likes used: picking w by variety does not leak test)
for w in [0.0, 2.0, 4.0, 8.0, 16.0]:
    movies = top10_movies(lambda u: cold_start_scores(u) + w * popularity_scaled, cold_users)
    print(f"w {w}: {len(np.unique(movies)):,} different movies · median rating count {np.median(rating_counts[movies]):,.0f}")

In [ ]:
# Cell 14: honest test Hit@10 of the chosen blend (w = 4, picked on val + variety)
BLEND_W = 4.0
print("cold start, blend w 4:", test_hit_at_10(lambda u: cold_start_scores(u) + BLEND_W * popularity_scaled, cold_users))

In [ ]:
# Cell 15: eyeball the app's lists for hand-made users (fold-in + blend w 4 + newer boost 0.3, like the app)
NEWER_BOOST = 0.3  # backend/app/core/config.py
movie_titles = pd.read_csv("/content/drive/MyDrive/tastelet/data/raw/ml-32m/movies.csv", usecols=["movieId", "title"])
idx_to_title = np.empty(n_movies, dtype=object)
for movie_id, title in zip(movie_titles["movieId"], movie_titles["title"]):
    idx = movie2idx.get(str(movie_id))
    if idx is not None:
        idx_to_title[idx] = title


def find_movie_idx(name: str, year: int) -> int:
    matches = [i for i in range(n_movies) if name.lower() in idx_to_title[i].lower() and movie_years[i] == year]
    assert len(matches) == 1, f"{name} ({year}) matched {[idx_to_title[i] for i in matches]}"
    return matches[0]


HAND_MADE_USERS = {
    "Marvel fan": [("Infinity War - Part II", 2019), ("Black Panther", 2017), ("Thor: Ragnarok", 2017),
                   ("Guardians of the Galaxy", 2014), ("Spider-Man: Into the Spider-Verse", 2018)],
    "horror fan": [("Get Out", 2017), ("Hereditary", 2018), ("Conjuring", 2013),
                   ("Conjuring 2", 2016), ("Quiet Place", 2018)],
    "romance fan": [("La La Land", 2016), ("Notebook", 2004), ("Pride & Prejudice", 2005),
                    ("About Time", 2013), ("Before Sunrise", 1995)],
}

for user, named_picks in HAND_MADE_USERS.items():
    picks = [find_movie_idx(name, year) for name, year in named_picks]
    user_vec = fold_in_user_vector(picks)
    with torch.no_grad():
        scores = (best_model.score_vec(user_vec.expand(n_movies, -1), all_movies)
                  + BLEND_W * popularity_scaled + NEWER_BOOST * freshness)
        scores[picks] = -torch.inf  # never recommend a pick
        top = torch.topk(scores, TOP_K).indices.tolist()
    print(f"\n{user}:")
    for idx in top:
        print(f"  {idx_to_title[idx]} · {rating_counts[idx]:,} ratings")

In [ ]:
names = "Get Out|Hereditary|Conjuring|Babadook|Quiet Place|La La Land|Notebook|Pride & Prejudice|About Time|Before Sunrise"
hits = movie_titles[movie_titles["title"].str.contains(names, case=False)]
hits = hits.assign(in_model=hits["movieId"].astype(str).isin(movie2idx))
print(hits[hits["in_model"]].to_string(index=False))

In [ ]:
# Cell 16: why no Marvel for the Marvel fan? where the picks' ratings went, and the list with/without the blend
val_counts = np.bincount(val["movie_idx"], minlength=n_movies)
test_counts = np.bincount(test["movie_idx"], minlength=n_movies)
marvel_picks = [find_movie_idx(name, year) for name, year in HAND_MADE_USERS["Marvel fan"]]
for idx in marvel_picks:
    print(f"{idx_to_title[idx]}: train {rating_counts[idx]:,} · val {val_counts[idx]:,} · test {test_counts[idx]:,}")

marvel_vec = fold_in_user_vector(marvel_picks)
with torch.no_grad():
    model_scores = best_model.score_vec(marvel_vec.expand(n_movies, -1), all_movies)
    print("pick scores:", [round(s, 2) for s in model_scores[marvel_picks].tolist()])
    for w in [0.0, BLEND_W]:
        scores = model_scores + w * popularity_scaled
        scores[marvel_picks] = -torch.inf
        print(f"\nw {w}:", " | ".join(idx_to_title[i] for i in torch.topk(scores, TOP_K).indices.tolist()))

In [ ]:
# Cell 17: is the fold-in stuck in dead ReLUs? (flat output ~0.6, zero gradient to escape)
print(f"test users with pick score < 1.0: {np.mean(np.array(pick_scores) < 1.0):.0%}")  # pick_scores from Cell 6

marvel_ids = torch.tensor(marvel_picks, device=DEVICE)


def describe(vec: torch.Tensor) -> str:
    with torch.no_grad():
        first_layer = best_model.mlp[0](torch.cat([vec.expand(len(marvel_picks), -1),
                                                    best_model.movie_embedding(marvel_ids)], dim=1))
        scores = best_model.score_vec(vec.expand(len(marvel_picks), -1), marvel_ids)
    return f"pick scores {[round(s, 2) for s in scores.tolist()]} · active hidden units {(first_layer > 0).float().mean().item():.0%}"


print("start (mean user):", describe(mean_user_vec))
for lr in [0.05, 0.01, 0.005]:
    print(f"lr {lr}, 300 steps:", describe(fold_in_user_vector(marvel_picks, steps=300, lr=lr)))

In [ ]:
# Cell 18: does the TRAINED model score these movies high for their real 5-star fans? Model problem vs fold-in problem
check_movies = marvel_picks + [find_movie_idx("Fight Club", 1999), find_movie_idx("Amelie", 2001)]
with torch.no_grad():
    for idx in check_movies:
        fans = train.loc[(train["movie_idx"] == idx) & (train["rating"] == 5.0), "user_idx"].values[:2_000]
        fan_ids = torch.as_tensor(fans, device=DEVICE, dtype=torch.long)
        fan_scores = best_model(fan_ids, torch.full_like(fan_ids, idx))
        negatives_per_positive = (N_NEGATIVES * len(train) * negative_weights[idx] / negative_weights.sum()
                                  / rating_counts[idx])  # expected training negatives per real rating (ADR 0002)
        print(f"{idx_to_title[idx]}: median score for 5-star fans {fan_scores.median().item():.2f} · "
              f"negatives per real rating {negatives_per_positive:.1f}")

In [ ]:
# Cell 19: start the fold-in from the picks' fans instead of the average user
FAN_CHUNK = 2_000_000  # look up embeddings in chunks so 25M x 32 floats never sit on the GPU at once
liked_train = train[train["rating"] >= LIKED_RATING]
liked_movies_t = torch.as_tensor(liked_train["movie_idx"].values, device=DEVICE, dtype=torch.long)
liked_users_t = torch.as_tensor(liked_train["user_idx"].values, device=DEVICE, dtype=torch.long)
fan_sums = torch.zeros(n_movies, EMBEDDING_DIM, device=DEVICE)
with torch.no_grad():
    for chunk_start in range(0, len(liked_train), FAN_CHUNK):
        chunk = slice(chunk_start, chunk_start + FAN_CHUNK)
        fan_sums.index_add_(0, liked_movies_t[chunk], best_model.user_embedding(liked_users_t[chunk]))
fan_counts = torch.bincount(liked_movies_t, minlength=n_movies)
fan_centroids = fan_sums / fan_counts.clamp(min=1).unsqueeze(1)
fan_centroids[fan_counts == 0] = mean_user_vec  # nobody liked it in train: fall back to the average user


def fan_start(picks: list[int]) -> torch.Tensor:
    return fan_centroids[picks].mean(dim=0, keepdim=True)


# 1) Eyeball: the same three hand-made users as Cell 15
for user, named_picks in HAND_MADE_USERS.items():
    picks = [find_movie_idx(name, year) for name, year in named_picks]
    vec = fold_in_user_vector(picks, start=fan_start(picks))
    with torch.no_grad():
        model_scores = best_model.score_vec(vec.expand(n_movies, -1), all_movies)
        pick_score = model_scores[picks].mean().item()
        scores = model_scores + BLEND_W * popularity_scaled + NEWER_BOOST * freshness
        scores[picks] = -torch.inf
        top = torch.topk(scores, TOP_K).indices.tolist()
    print(f"\n{user} (pick score {pick_score:.2f}):\n  " + "\n  ".join(idx_to_title[i] for i in top))

# 2) Val Hit@10 on the same 1,000 val users as Cell 11 (there: w0 0.142, w4 0.199, w8 0.213, popularity 0.200)
fan_vecs = {u: fold_in_user_vector(tune_picks[u], start=fan_start(tune_picks[u])) for u in TUNE_USERS}
for w in [0.0, 4.0, 8.0]:
    hit = hit_at_10(lambda u: best_model.score_vec(fan_vecs[u].expand(n_movies, -1), all_movies)
                    + w * popularity_scaled, TUNE_USERS)
    print(f"fan start, w {w}: val Hit@10 {hit:.3f}")

In [ ]:
# Cell 20: fan start, final check ON TEST (once): Hit@10 and variety at w 4
BLEND_W = 4.0  # chosen in Cell 11/13; unchanged
started = time.time()
cold_fan_vecs = {u: fold_in_user_vector(picks_by_user[u], start=fan_start(picks_by_user[u])) for u in cold_users}
print(f"folded in {len(cold_fan_vecs):,} users in {time.time() - started:.0f}s")


def fan_cold_scores(u):
    return best_model.score_vec(cold_fan_vecs[u].expand(n_movies, -1), all_movies) + BLEND_W * popularity_scaled


print("test Hit@10, fan start, w 4:", test_hit_at_10(fan_cold_scores, cold_users))
fan_top = top10_movies(fan_cold_scores, cold_users)
print(f"variety: {len(np.unique(fan_top)):,} different movies · median rating count {np.median(rating_counts[fan_top]):,.0f}")

In [ ]:
# Cell 21: personas — K-means (K = 6, fixed decision) on L2-normalized v3 user vectors, genres ranked by lift
from sklearn.cluster import KMeans

PERSONA_K, N_TOP_GENRES, N_TOP_MOVIES = 6, 3, 5
MIN_GENRE_LIKES = 30  # as notebook 04: ignore tiny genres whose lift is noise
MIN_CLUSTER_RATERS = 1_000  # notebook 04 used 10 with ~100 users per cluster; v3 has ~33k
NOT_A_GENRE = {"IMAX", "(no genres listed)"}

user_vecs = best_model.user_embedding.weight.cpu().numpy()
unit_user_vecs = user_vecs / np.linalg.norm(user_vecs, axis=1, keepdims=True)
started = time.time()
persona_kmeans = KMeans(n_clusters=PERSONA_K, n_init=10, random_state=SEED).fit(unit_user_vecs)
user_cluster = persona_kmeans.labels_
print(f"K-means in {time.time() - started:.0f}s · cluster sizes {np.bincount(user_cluster).tolist()}")

# Per (cluster, movie) instead of per rating: 25M ratings x genres would not fit in Colab's RAM.
per_movie = (train.assign(cluster=user_cluster[train["user_idx"].values], liked=train["rating"] >= LIKED_RATING)
             .groupby(["cluster", "movie_idx"])["liked"].agg(["size", "sum"]).reset_index())
movie_genres = pd.read_csv("/content/drive/MyDrive/tastelet/data/raw/ml-32m/movies.csv", usecols=["movieId", "genres"])
movie_genres["movie_idx"] = movie_genres["movieId"].astype(str).map(movie2idx)
movie_genres = movie_genres.dropna(subset=["movie_idx"]).astype({"movie_idx": int})
genre_rows = movie_genres.assign(genre=movie_genres["genres"].str.split("|")).explode("genre")[["movie_idx", "genre"]]
genre_rows = genre_rows[~genre_rows["genre"].isin(NOT_A_GENRE)]
genre_likes = per_movie.merge(genre_rows, on="movie_idx").groupby(["cluster", "genre"])["sum"].sum().unstack(fill_value=0)
overall_share = genre_likes.sum(axis=0) / genre_likes.to_numpy().sum()
genre_lift = genre_likes.div(genre_likes.sum(axis=1), axis=0) / overall_share

top_genres = {c: genre_lift.loc[c][genre_likes.loc[c] >= MIN_GENRE_LIKES].nlargest(N_TOP_GENRES) for c in range(PERSONA_K)}
look_alikes = [(a, b) for a in range(PERSONA_K) for b in range(a + 1, PERSONA_K)
               if len(set(top_genres[a].index) & set(top_genres[b].index)) >= 2]
print("look-alike pairs (share 2 of top 3 genres):", look_alikes or "none")

for c in range(PERSONA_K):
    rated = per_movie[(per_movie["cluster"] == c) & (per_movie["size"] >= MIN_CLUSTER_RATERS)]
    best = rated.assign(like_rate=rated["sum"] / rated["size"]).nlargest(N_TOP_MOVIES, "like_rate")
    print(f"\ncluster {c} ({np.bincount(user_cluster)[c]:,} users): {top_genres[c].round(2).to_dict()}")
    print("  " + " | ".join(idx_to_title[i] for i in best["movie_idx"]))

# Which cluster do the hand-made users land in? (fan start, cosine to the normalized centroids, like the app)
unit_centroids = persona_kmeans.cluster_centers_ / np.linalg.norm(persona_kmeans.cluster_centers_, axis=1, keepdims=True)
for user, named_picks in HAND_MADE_USERS.items():
    picks = [find_movie_idx(name, year) for name, year in named_picks]
    vec = fold_in_user_vector(picks, start=fan_start(picks)).cpu().numpy().ravel()
    print(f"{user} -> cluster {int(np.argmax(unit_centroids @ (vec / np.linalg.norm(vec))))}")

In [ ]:
# Cell 22: does K = 6 still fit v3? Per K: look-alike pairs, smallest cluster, where the 3 hand-made users land
genre_names = sorted(genre_rows["genre"].unique())
genre_matrix = np.zeros((n_movies, len(genre_names)))
genre_matrix[genre_rows["movie_idx"].values, pd.Categorical(genre_rows["genre"], categories=genre_names).codes] = 1
is_liked = (train["rating"] >= LIKED_RATING).values
liked_user_idx, liked_movie_idx = train["user_idx"].values[is_liked], train["movie_idx"].values[is_liked]


def cluster_likes(labels: np.ndarray, k: int) -> np.ndarray:
    """Liked ratings per (cluster, movie), shape (k, n_movies)."""
    return np.bincount(labels[liked_user_idx] * n_movies + liked_movie_idx, minlength=k * n_movies).reshape(k, n_movies)


def genre_lift_of(likes: np.ndarray) -> np.ndarray:
    genre_likes = likes @ genre_matrix
    lift = (genre_likes / genre_likes.sum(axis=1, keepdims=True)) / (genre_likes.sum(axis=0) / genre_likes.sum())
    return np.where(genre_likes >= MIN_GENRE_LIKES, lift, 0.0)


hand_made_unit_vecs = {}
for user, named_picks in HAND_MADE_USERS.items():
    picks = [find_movie_idx(name, year) for name, year in named_picks]
    vec = fold_in_user_vector(picks, start=fan_start(picks)).cpu().numpy().ravel()
    hand_made_unit_vecs[user] = vec / np.linalg.norm(vec)

kmeans_by_k = {}
for k in range(6, 11):
    kmeans_by_k[k] = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(unit_user_vecs)
    labels = kmeans_by_k[k].labels_
    top3 = [set(np.argsort(-row)[:N_TOP_GENRES]) for row in genre_lift_of(cluster_likes(labels, k))]
    look_alikes = sum(len(top3[a] & top3[b]) >= 2 for a in range(k) for b in range(a + 1, k))
    centroids = kmeans_by_k[k].cluster_centers_ / np.linalg.norm(kmeans_by_k[k].cluster_centers_, axis=1, keepdims=True)
    landed = [int(np.argmax(centroids @ v)) for v in hand_made_unit_vecs.values()]
    print(f"K {k}: look-alike pairs {look_alikes} · smallest cluster {np.bincount(labels).min():,} · "
          f"Marvel/horror/romance -> {landed}")